In [34]:
# Imports and setup
import sys
from pathlib import Path
sys.path.append(str(Path('../../src').resolve()))

import os, random
import pandas as pd, json, time, traceback, ast
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (LSTM, Dense, Dropout, Bidirectional,
                                     Conv1D, Flatten, Input, Attention, GlobalAveragePooling1D)
from tensorflow.keras import backend as K

# Importa tus configs y utils
from utils.lstm_configs import LSTM_CONFIGS
from utils.variants import VARIANT_DEFS, apply_variant
from utils.eval_metrics import evaluate_predictions

SEED = 1
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


BEST_IN = Path("../../data/out/best_models_lstm/in/best_models_lstm.xlsx")
BEST_OUT = Path("../../data/out/best_models_lstm")
MODELS_DIR = BEST_OUT / "models"
PLOTS_DIR = BEST_OUT / "plots"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

categorical_numeric=["YEAR","MONTH","DAY","HORA","NIVEL_ENSO","DIA_SEMANA","FESTIVO"]

BASE_DATA_PATH = Path('../../data/out/dataset_final.csv')
VARIANTS = [(v['name'], v) for v in VARIANT_DEFS]

for vname, vdef in VARIANTS:
    print(vname, '-> base data exists?', BASE_DATA_PATH.exists())


v1_original -> base data exists? True
v1_original_lags -> base data exists? True
v2_with_calendar -> base data exists? True
v2_with_calendar_lags -> base data exists? True
v3_no_solar -> base data exists? True
v3_no_solar_lags -> base data exists? True
v4_no_fuel_consumption -> base data exists? True
v4_no_fuel_consumption_lags -> base data exists? True
v5_no_fuel_and_cost -> base data exists? True
v5_no_fuel_and_cost_lags -> base data exists? True
v6_no_econ_fuel_cost -> base data exists? True
v6_no_econ_fuel_cost_lags -> base data exists? True
v7_only_gen_enso -> base data exists? True
v7_only_gen_enso_lags -> base data exists? True
v8_only_gen_no_solar_enso -> base data exists? True
v8_only_gen_no_solar_enso_lags -> base data exists? True
v9_date_range_all -> base data exists? True
v9_date_range_all_lags -> base data exists? True
v10_date_range_no_solar -> base data exists? True
v10_date_range_no_solar_lags -> base data exists? True


In [35]:
# Buscar config por nombre
def get_lstm_config(name):
    for c in LSTM_CONFIGS:
        if c["name"] == name:
            return c.copy()
    raise ValueError(f"Config {name} no encontrada en LSTM_CONFIGS")


def _prepare_variant_data_for_lstm(df, target_column='PRECIO', timesteps=24,
                                   categorical_numeric=["YEAR","MONTH","DAY","HORA","NIVEL_ENSO","DIA_SEMANA","FESTIVO"]):
    df = df.drop(columns=['FECHA_HORA'], errors='ignore')

    # Separate numeric vs categorical
    numeric_cols = [c for c in df.columns if c not in categorical_numeric + [target_column]]
    X_num = df[numeric_cols].values.astype('float32') if numeric_cols else None
    X_cat = df[[c for c in categorical_numeric if c in df.columns]].values.astype('float32')
    y = df[target_column].values.astype('float32')

    # Scale only numeric features
    if X_num is not None:
        from sklearn.preprocessing import MinMaxScaler
        scaler = MinMaxScaler()
        X_num_scaled = scaler.fit_transform(X_num)
        X_all = np.concatenate([X_num_scaled, X_cat], axis=1)
    else:
        X_all = X_cat

    # Build sequences
    X_seq, y_seq = [], []
    for i in range(len(X_all) - timesteps):
        X_seq.append(X_all[i:i+timesteps, :])
        y_seq.append(y[i+timesteps])
    X_seq, y_seq = np.array(X_seq), np.array(y_seq)

    train_size = int(len(X_seq) * 0.8)
    return X_seq[:train_size], X_seq[train_size:], y_seq[:train_size], y_seq[train_size:]
    
def build_lstm(input_shape, config):
    """
    Build an LSTM model according to the given configuration.
    Supports: stacked LSTM, bidirectional, CNN+LSTM, Attention.
    """

    model = None  # placeholder

    # CNN + LSTM case (Sequential)
    if "conv1d" in config:
        model = Sequential()
        conv = config["conv1d"]
        model.add(Conv1D(filters=conv["filters"],
                         kernel_size=conv["kernel_size"],
                         activation=conv["activation"],
                         input_shape=input_shape))
        model.add(Flatten())
        for units in config["lstm_layers"]:
            model.add(Dense(units, activation="relu"))
        if config.get("dropout", 0) > 0:
            model.add(Dropout(config["dropout"]))
        model.add(Dense(1))

    # Attention case (Functional API)
    elif config.get("attention", False):
        inputs = Input(shape=input_shape)
        x = LSTM(config["layers"][0], return_sequences=True)(inputs)
        context = Attention()([x, x])
        context = GlobalAveragePooling1D()(context) 

        if config.get("dropout", 0) > 0:
            context = Dropout(config["dropout"])(context)
        outputs = Dense(1)(context)
        model = Model(inputs, outputs)


    # Standard LSTM / BiLSTM (Sequential)
    else:
        model = Sequential()
        for i, units in enumerate(config["layers"]):
            return_sequences = i < len(config["layers"]) - 1
            if config.get("bidirectional", False):
                model.add(Bidirectional(
                    LSTM(units,
                         return_sequences=return_sequences,
                         recurrent_dropout=config.get("recurrent_dropout", 0.0)),
                    input_shape=input_shape))
            else:
                model.add(LSTM(units,
                               return_sequences=return_sequences,
                               recurrent_dropout=config.get("recurrent_dropout", 0.0),
                               input_shape=input_shape))
            if config.get("dropout", 0) > 0:
                model.add(Dropout(config["dropout"]))
        model.add(Dense(1))

    # Compile once at the end
    model.compile(optimizer=config["optimizer"], loss="mse")
    return model

def run_and_save_best_lstm(config, variant, df, timesteps=24):
    X_train, X_test, y_train, y_test = _prepare_variant_data_for_lstm(df, timesteps=timesteps, categorical_numeric=categorical_numeric)

    result = {"model": "LSTM", "config": config["name"], "variant": variant}
    start_time = time.time()

    try:
        model = build_lstm((X_train.shape[1], X_train.shape[2]), config)
        history = model.fit(
            X_train, y_train,
            validation_data=(X_test, y_test),
            epochs=config["epochs"],
            batch_size=config["batch_size"],
            verbose=0
        )

        y_pred = model.predict(X_test).ravel()
        metrics = evaluate_predictions(y_test, y_pred)
        result.update(metrics)
        result["status"] = "trained"

        # Guardar modelo y history
        model.save(MODELS_DIR / f"{variant}_{config['name']}.h5")
        with open(MODELS_DIR / f"{variant}_{config['name']}_history.json", "w") as f:
            json.dump(history.history, f)

        # --- Gráficas ---
        # 1. Loss
        plt.plot(history.history["loss"], label="train")
        plt.plot(history.history["val_loss"], label="val")
        plt.legend(); plt.tight_layout()
        plt.savefig(PLOTS_DIR / f"loss_{variant}_{config['name']}.png"); plt.close()

        # 2. Predicciones vs reales
        plt.figure(figsize=(12,5))
        plt.plot(y_test[:200], label="Real")
        plt.plot(y_pred[:200], label="Predicho")
        plt.legend(); plt.tight_layout()
        plt.savefig(PLOTS_DIR / f"pred_{variant}_{config['name']}.png"); plt.close()

        # 3. Residuos
        resid = y_test - y_pred
        plt.figure(figsize=(12,4))
        plt.plot(resid[:200], color="tab:red")
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / f"resid_{variant}_{config['name']}.png"); plt.close()

        # 4. Scatter con línea y=x
        plt.figure(figsize=(5,5))
        plt.scatter(y_test, y_pred, s=6, alpha=0.5)
        lims = [min(plt.xlim()[0], plt.ylim()[0]), max(plt.xlim()[1], plt.ylim()[1])]
        plt.plot(lims, lims, 'r--', linewidth=2)
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / f"scatter_{variant}_{config['name']}.png"); plt.close()

    except Exception as e:
        tb = traceback.format_exc()
        result["status"] = "error"
        result["error"] = str(e)
        result["traceback"] = tb

    finally:
        elapsed = time.time() - start_time
        result["n_total"] = len(y_train) + len(y_test)
        result["n_train"] = len(y_train)
        result["n_test"] = len(y_test)
        result["train_time_s"] = elapsed

    return result

In [36]:
# --- MAIN ---
best_df = pd.read_excel(BEST_IN)
best_df.head()

,variant,model,config,R2,MAE,MSE,RMSE,MAPE(%),status,n_total,n_train,n_test,train_time_s,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16
0,v1_original,LSTM,cnn_lstm,0.603645,180.974380,99650.476562,315.674652,123.102820,trained,48168,38534,9634,80.602915,NaN,NaN,NaN,NaN
1,v1_original_lags,LSTM,cnn_lstm,0.877402,82.915123,30839.160156,175.610825,165.548527,trained,48144,38515,9629,144.987116,NaN,NaN,NaN,NaN
2,v2_with_calendar,LSTM,cnn_lstm,0.409195,226.943253,148538.671875,385.407166,98.911065,trained,48168,38534,9634,132.972647,NaN,NaN,NaN,347.02
3,v2_with_calendar_lags,LSTM,cnn_lstm,0.870367,86.073517,32608.923828,180.579407,164.801240,trained,48144,38515,9629,139.015848,NaN,NaN,NaN,NaN
4,v3_no_solar,LSTM,cnn_lstm,0.256702,271.460022,186877.984375,432.293854,85.815006,trained,48168,38534,9634,134.537482,NaN,NaN,NaN,NaN


In [37]:
rows = []

for _, row in best_df.iterrows():
    variant = row["variant"]
    config_name = row["config"]
    vdef = next(v for v in VARIANT_DEFS if v["name"] == variant)

    try:
        df, _ = apply_variant(pd.read_csv(BASE_DATA_PATH), vdef, date_col="FECHA_HORA")
        config = get_lstm_config(config_name)
        res = run_and_save_best_lstm(config, variant, df, timesteps=24)
        rows.append(res)
        print(f"OK {variant}-{config_name}: MAE={res.get('MAE'):.2f}")
    except Exception as e:
        print(f"ERROR {variant}-{config_name}: {e}")
        traceback.print_exc()

# Consolidar métricas
if rows:
    df_out = pd.DataFrame(rows)
    df_out.to_excel(BEST_OUT / "best_lstm_metrics.xlsx", index=False)

302/302 [==============================] - 1s 2ms/step


c:\Users\Camilo\anaconda3\envs\env_tf\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


OK v1_original-cnn_lstm: MAE=223.88


KeyboardInterrupt: 